# Complementary (Grounded) Concentric Transmon
## Full Design, Simulation and Quantum Analysis — Qiskit-Metal + pyEPR
### Version 9 -- Bug fixes: mpl renderer, analyze_setup, save_screenshot, f_hfss fallback, energy_elec Dict, Ej addict fix

### Geometry (v2 — explicit metallic plane)

```
        ┌─────────────────────────────────┐
        │           W x W                 │
        │       metallic plane            │
        │                                 │
        │        ╭──────────╮  ←Wslot     │
        │      ╭─╯  ○ disk  ╰─╮           │
        │  ──JJ─  Rdisk   JJ──            │
        │      ╰─╮          ╭─╯           │
        │        ╰──────────╯             │
        └─────────────────────────────────┘
```

| Parameter | Description | Design knob |
|---|---|---|
| `W` | Side of outer metallic square | Isolation / boundary condition |
| `Rdisk` | Radius of central metallic disk | Primary Ec knob (↑Rdisk → ↓Ec) |
| `Wslot` | Width of circular slot (gap) | Secondary Ec knob (↑Wslot → ↓Ec) |
| `jj_width` | Tangential width of each JJ bridge | JJ area / Ic spread |
| `Lj` | Josephson inductance (HFSS variable) | Ej → f_01 |

### Design targets
| Parameter | Target |
|---|---|
| f_01 | 3 – 4.8 GHz |
| Ec | ~140 MHz |
| Ej/Ec | > 50 |

### Reference: rfphotoniqlabs/qiskit-metal
- `tutorials/4 Analysis/A. Core/4.02 Eigenmode and EPR.ipynb`


In [1]:
%reload_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq

import qiskit_metal as metal
from qiskit_metal import designs, draw
from qiskit_metal import MetalGUI, Dict
from qiskit_metal.toolbox_python.attr_dict import Dict
from qiskit_metal.qlibrary.core import QComponent

print(f"Qiskit-Metal version: {metal.__version__}")


Qiskit-Metal version: 0.5.3.post1


12:07PM 50s WARNING [_qt_message_handler]: WARNING: "Unable to open monitor interface to \\\\.\\DISPLAY513:" "Unknown error 0xe0000225." (No context available from Qt)
Python Traceback (most recent call last):
  File "C:\Users\Kobi\anaconda3\envs\quantum_metal_env\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\Kobi\anaconda3\envs\quantum_metal_env\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\Kobi\anaconda3\envs\quantum_metal_env\Lib\site-packages\ipykernel\kernelapp.py", line 758, in start
    self.io_loop.start()
  File "C:\Users\Kobi\anaconda3\envs\quantum_metal_env\Lib\site-packages\tornado\platform\asyncio.py", line 211, in start
    self.asyncio_loop.run_forever()
  File "C:\Users\Kobi\anaconda3\envs\quantum_metal_env\Lib\asyncio\base_events.py", line 608, in run_forever
    self._run_once()
  File "C:\Users\Kobi\anaconda3\envs\quantum_metal_env\Lib\

---
## Step 1 — Load `CircmonG` (single-JJ grounded circular transmon)

The `CircmonG` component is imported from the user-components library.

**Geometry**
- `pad`      : filled circle of radius `pad_r` — the qubit island
- `pocket`   : annular ring of width `pad2pk_gap`, subtracted from the ground plane
- `rect_jj1` : **single** Josephson Junction at azimuthal angle `jj_angle`

| Option | Default | Description |
|---|---|---|
| `pad_r` | `'100um'` | Qubit island radius |
| `pad2pk_gap` | `'100um'` | Circular slot width |
| `jj_width` | `'20um'` | JJ bridge width |
| `jj_angle` | `0` | JJ angle in degrees (0 = east) |
| `jj_in_pad` | `'2um'` | JJ overlap into pad edge |
| `jj_in_pk` | `'2um'` | JJ overlap into pocket edge |

**HFSS junction object names** (auto-generated by Qiskit-Metal renderer):
```
rect : JJ_rect_Lj_{component_name}_rect_jj1
line : JJ_Lj_{component_name}_rect_jj1_
```


In [2]:
from qiskit_metal.qlibrary.user_components.circmong_singleJJ import CircmonG
print("CircmonG imported successfully.")
print()
print("Junction key in HFSS: 'rect_jj1'")
print("For component name 'TC1':")
print("  rect : JJ_rect_Lj_TC1_rect_jj1")
print("  line : JJ_Lj_TC1_rect_jj1_")


CircmonG imported successfully.

Junction key in HFSS: 'rect_jj1'
For component name 'TC1':
  rect : JJ_rect_Lj_TC1_rect_jj1
  line : JJ_Lj_TC1_rect_jj1_


---
## Step 2 — Build the Design and Display in MetalGUI


In [3]:
# ── Create design ─────────────────────────────────────────────────────────
# Set chip size to match W (the metallic plane side).
# A small border is added so the HFSS airbox does not clip the component.
W_um = 800    # must match the W option below
H_um = 280 # substrate height

design = designs.DesignPlanar({}, True)
design.chips.main.size['size_x'] = f'{W_um}um'
design.chips.main.size['size_y'] = f'{W_um}um'
design.chips.main.size['size_z'] = f'{-H_um}um'
design.chips.main.size['center_x'] = '0um'
design.chips.main.size['center_y'] = '0um'

gui = MetalGUI(design)
print("Design created. MetalGUI opened.")


Design created. MetalGUI opened.


In [4]:
from qiskit_metal.qlibrary.user_components.circmong_singleJJ import CircmonG

circmon1 = CircmonG(design, 'TC1', options=dict(
    pos_x      = '0mm',
    pos_y      = '0mm',
    pad_r      = '200um',   # qubit island radius — primary Ec knob
    pad2pk_gap = '100um',   # slot width          — secondary Ec knob
    jj_width   = '20um',
    jj_angle   = 0,         # 0 deg = east (+x); single JJ only
    jj_in_pad  = '2um',
    jj_in_pk   = '2um',
))


In [5]:
design.overwrite_enabled = True
gui.rebuild()
gui.autoscale()

print("CircmonG built.")
print()
print(f"  pad_r      : {circmon1.options.pad_r}")
print(f"  pad2pk_gap : {circmon1.options.pad2pk_gap}")
print(f"  jj_width   : {circmon1.options.jj_width}")
print(f"  jj_angle   : {circmon1.options.jj_angle} deg")
print()
print("Junction table:")
print(design.qgeometry.tables['junction'][
    ['component','name','width','hfss_inductance','hfss_capacitance']])


CircmonG built.

  pad_r      : 200um
  pad2pk_gap : 100um
  jj_width   : 20um
  jj_angle   : 0 deg

Junction table:
  component      name  width hfss_inductance hfss_capacitance
0         1  rect_jj1   0.02              Lj               Cj


In [6]:
# -- Verify geometry using matplotlib renderer -------------------------------
# QM 0.5.x: design.renderers.mpl may be a Dict placeholder if Qt GUI is used.
# Fallback to per-component plotting when mpl renderer is unavailable.
try:
    import inspect
    _mpl = design.renderers.mpl
    if not callable(getattr(_mpl, 'render', None)):
        raise TypeError("mpl.render is not callable (addict Dict placeholder)")
    fig, ax = plt.subplots(1, 1, figsize=(6, 6))
    _mpl.render(ax=ax)
    ax.set_title('CircmonG -- Single JJ Layout', fontsize=12)
    ax.set_xlabel('x (mm)')
    ax.set_ylabel('y (mm)')
    plt.tight_layout()
    plt.savefig('circmong_layout.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Layout saved: circmong_layout.png")
except Exception as e:
    print(f"  [mpl renderer unavailable: {e}]")
    print("  Falling back to component qgeometry plot...")
    try:
        fig, ax = plt.subplots(1, 1, figsize=(6, 6))
        for name, comp in design.components.items():
            comp.qgeometry_plot(ax=ax)
        ax.set_title('CircmonG -- Single JJ Layout (qgeometry)', fontsize=12)
        ax.autoscale()
        plt.tight_layout()
        plt.show()
    except Exception as e2:
        print(f"  [qgeometry plot also failed: {e2}]")
        print("  Use the Metal GUI (gui.autoscale()) to inspect the layout.")


  [mpl renderer unavailable: mpl.render is not callable (addict Dict placeholder)]
  Falling back to component qgeometry plot...


In [7]:
# ── Analytical preview: f_01 and Ej/Ec vs Lj (fixed Ec = 140 MHz) ────────
# Single JJ: Ej_eff = Ej_single  (no parallel combination)
h_planck = 6.626e-34
Phi0     = 2.068e-15

def Lj_to_Ej_GHz(Lj_nH):
    """Single-JJ Ej in GHz from Lj in nH."""
    return (Phi0/(2*np.pi))**2 / (Lj_nH*1e-9) / (h_planck*1e9)

def f01_approx(Ej_GHz, Ec_GHz):
    """f_01 = sqrt(8*Ej*Ec) - Ec  [transmon approximation]."""
    return np.sqrt(8*Ej_GHz*Ec_GHz) - Ec_GHz

Ec = 0.140   # GHz — target Ec
print(f"Analytical preview  |  Ec = {Ec*1e3:.0f} MHz  |  single JJ")
print(f"{'Lj (nH)':>10} {'Ej (GHz)':>12} {'Ej/Ec':>8} {'f_01 (GHz)':>12}")
print("-" * 48)
for Lj_nH in [8, 9, 10, 11, 12, 13, 14, 15, 20, 25, 30]:
    Ej   = Lj_to_Ej_GHz(Lj_nH)   # single JJ — no x2
    ratio = Ej / Ec
    f01   = f01_approx(Ej, Ec)
    tag   = "  <-- target" if 3.0 <= f01 <= 4.8 else ""
    print(f"{Lj_nH:>10.0f} {Ej:>12.3f} {ratio:>8.0f} {f01:>12.4f}{tag}")


Analytical preview  |  Ec = 140 MHz  |  single JJ
   Lj (nH)     Ej (GHz)    Ej/Ec   f_01 (GHz)
------------------------------------------------
         8       20.436      146       4.6442  <-- target
         9       18.166      130       4.3706  <-- target
        10       16.349      117       4.1391  <-- target
        11       14.863      106       3.9400  <-- target
        12       13.624       97       3.7663  <-- target
        13       12.576       90       3.6130  <-- target
        14       11.678       83       3.4765  <-- target
        15       10.899       78       3.3539  <-- target
        20        8.174       58       2.8858
        25        6.540       47       2.5663
        30        5.450       39       2.3305


---
## Step 3 — Render to HFSS and Configure Josephson Junctions

### What `sim.run()` does in HFSS
1. Creates a new eigenmode design named `Qbit_hfss`
2. Exports `metal_plane` (W×W with hole) → HFSS PEC sheet on substrate top face
3. Exports `disk` → HFSS PEC sheet (the qubit island)
4. Creates silicon substrate box (εr = 11.7) sized to the chip
5. Exports `jj_east` / `jj_west` → HFSS lumped RLC elements
6. Assigns `Lj` and `Cj` as HFSS design variables on each lumped element

### JJ HFSS object naming convention
Qiskit-Metal's HFSS renderer auto-generates names from the component name and
the key passed to `add_qgeometry('junction', {key: geom})`:
```
rect : JJ_rect_Lj_{component_name}_{key}
line : JJ_Lj_{component_name}_{key}_
```
For component `CCT1` with keys `jj_east` and `jj_west`:
```
jj_east rect : JJ_rect_Lj_CCT1_jj_east
jj_east line : JJ_Lj_CCT1_jj_east_
jj_west rect : JJ_rect_Lj_CCT1_jj_west
jj_west line : JJ_Lj_CCT1_jj_west_
```


In [8]:
from qiskit_metal.analyses.quantization import EPRanalysis

# ── Create EPRanalysis object ─────────────────────────────────────────────
eig_qb = EPRanalysis(design, "hfss")

# ── Simulation setup ──────────────────────────────────────────────────────
eig_qb.sim.setup.name          = 'Qbit_Setup'
eig_qb.sim.setup.n_modes       = 1
eig_qb.sim.setup.max_passes    = 15
eig_qb.sim.setup.max_delta_f   = 0.1    # convergence: 0.1% change in freq
eig_qb.sim.setup.min_freq_ghz  = 1.0
eig_qb.sim.setup.min_converged = 2

# ── Junction variable (single JJ) ─────────────────────────────────────────
# Single JJ: Ej = (Phi0/2pi)^2 / Lj  (no parallel combination factor)
# For f_01 ~ 3.5 GHz at Ec ~ 140 MHz: Ej ~ 11.8 GHz -> Lj ~ 13.5 nH
# Adjust Lj to tune f_01.
eig_qb.sim.setup.vars = Dict(
    Lj = '13 nH',   # single JJ inductance — adjust to tune f_01
    Cj = '2 fF',    # shunt capacitance
)

# ── HFSS box buffer ───────────────────────────────────────────────────────
eig_qb.sim.renderer.options['x_buffer_width_mm'] = 0.5
eig_qb.sim.renderer.options['y_buffer_width_mm'] = 0.5

print("EPRanalysis setup:")
eig_qb.sim.setup


EPRanalysis setup:


{'name': 'Qbit_Setup',
 'reuse_selected_design': True,
 'reuse_setup': True,
 'min_freq_ghz': 1.0,
 'n_modes': 1,
 'max_delta_f': 0.1,
 'max_passes': 15,
 'min_passes': 1,
 'min_converged': 2,
 'pct_refinement': 30,
 'basis_order': 1,
 'vars': {'Lj': '13 nH', 'Cj': '2 fF'}}

In [9]:
# ── Render design to HFSS ─────────────────────────────────────────────────
# Prerequisite: Ansys HFSS must be running with a valid licence.

eig_qb.sim.run(
    name='Qbit_hfss',
    components=['TC1'],
    open_terminations=[],
    box_plus_buffer=True,
)

eig_qb.sim.print_run_args()
print()
print("HFSS rendering complete.")
print()

# ── Confirm JJ object names that pyEPR will look for ─────────────────────
comp = 'TC1'
print(f"Expected HFSS JJ object names for component '{comp}':")
print(f"  rect : JJ_rect_Lj_{comp}_rect_jj1")
print(f"  line : JJ_Lj_{comp}_rect_jj1_")


INFO 03:32PM [connect_project]: Connecting to Ansys Desktop API...
INFO 03:32PM [load_ansys_project]: 	Opened Ansys App
INFO 03:32PM [load_ansys_project]: 	Opened Ansys Desktop v2024.2.0
INFO 03:32PM [load_ansys_project]: 	Opened Ansys Project
	Folder:    C:/Users/Kobi/Documents/Ansoft/
	Project:   Project32
INFO 03:32PM [connect_design]: No active design found (or error getting active design).
INFO 03:32PM [connect]: 	 Connected to project "Project32". No design detected
INFO 03:32PM [connect_design]: 	Opened active design
	Design:    Qbit_hfss_hfss [Solution type: Eigenmode]
WARNING 03:32PM [connect_setup]: 	No design setup detected.
WARNING 03:32PM [connect_setup]: 	Creating eigenmode default setup.
INFO 03:32PM [get_setup]: 	Opened setup `Setup`  (<class 'pyEPR.ansys.HfssEMSetup'>)
INFO 03:32PM [get_setup]: 	Opened setup `Qbit_Setup`  (<class 'pyEPR.ansys.HfssEMSetup'>)
INFO 03:32PM [analyze]: Analyzing setup Qbit_Setup
03:35PM 52s INFO [get_f_convergence]: Saved convergences to C:

This analysis object run with the following kwargs:
{'name': 'Qbit_hfss', 'components': ['TC1'], 'open_terminations': [], 'port_list': None, 'jj_to_port': None, 'ignored_jjs': None, 'box_plus_buffer': True}


HFSS rendering complete.

Expected HFSS JJ object names for component 'TC1':
  rect : JJ_rect_Lj_TC1_rect_jj1
  line : JJ_Lj_TC1_rect_jj1_


---
## Step 3b -- Create CircmonG SPR EPR Boxes in HFSS

Surface Participation Ratio (SPR) analysis requires **thin reference boxes**
at each metal/dielectric interface. The boxes are **non-model** in HFSS (they
do not affect the eigenmode field solution) and serve only as integration
domains for the HFSS field calculator in Step 8b.

### Three interfaces for CircmonG (pad_r=200 um, pad2pk_gap=100 um)

| Box name | Interface | t_box | Region |
|---|---|---|---|
| `MA_island_TC1` | Metal-Air (island) | 5 nm | Square over circular island, side = 2 x pad_r |
| `SDA_slot_TC1` | Substrate-Air (slot) | 3 nm | Square over annular slot, side = 2*(pad_r+gap) |
| `MA_ground_TC1` | Metal-Air (ground) | 5 nm | Square ring, side = 4*(pad_r+gap) |

Square bounding boxes approximate the circular geometry.
An **area correction factor** (annular/circular area / bounding-square area)
is applied analytically in Step 8b.

> **Note:** Run this cell *after* `eig_qb.sim.run()` (Step 3) so that HFSS is
> open and the renderer modeler is accessible.  Since boxes are non-model they
> may also be added *after* the eigenmode solve without re-running.


In [10]:
# -- Step 3b: Create CircmonG-specific SPR EPR Boxes --------------------------
#
# Adapted from epr_box_automation.py for the CIRCULAR geometry of CircmonG.
# Original script uses TransmonPocket options (pad_width, pad_height, pad_gap).
# Here we use CircmonG options: pad_r and pad2pk_gap.
#
# Prerequisite: eig_qb.sim.run() must have been called so HFSS is open.

# -- Helpers: set CreateBox parameters via HFSS COM modeler -------------------
def _set_box_prop(design, obj_name, prop_path, value):
    """Set a property on CreateBox:1 child of an HFSS object via COM."""
    design.renderers.hfss.modeler._modeler \
        .GetChildObject(obj_name) \
        .GetChildObject('CreateBox:1') \
        .SetPropValue(prop_path, value)

def _set_obj_prop(design, obj_name, prop, value):
    """Set a top-level property on an HFSS object via COM."""
    design.renderers.hfss.modeler._modeler \
        .GetChildObject(obj_name) \
        .SetPropValue(prop, value)

def _draw_spr_box(design, name, x0, y0, xs, ys, thickness):
    """
    Draw a thin (nm-scale) non-model box centred at z=0 for SPR integration.

    Parameters
    ----------
    x0, y0     : str  Lower-left corner (HFSS parametric expressions)
    xs, ys     : str  XY dimensions
    thickness  : str  Z-dimension (box is centred at z=0)
    """
    modeler = design.renderers.hfss.modeler
    modeler.draw_box_center([0, 0, 0], [1e-3, 1e-3, 1e-3], name=name)
    _set_obj_prop(design, name, 'Model', False)          # non-model
    _set_box_prop(design, name, 'ZSize',      thickness)
    _set_box_prop(design, name, 'Position/Z', f'-({thickness})/2')
    _set_box_prop(design, name, 'XSize',      xs)
    _set_box_prop(design, name, 'Position/X', x0)
    _set_box_prop(design, name, 'YSize',      ys)
    _set_box_prop(design, name, 'Position/Y', y0)
    return name


def create_circmong_spr_boxes(design, q):
    """
    Create three thin EPR reference boxes for SPR surface-loss analysis.

    Boxes (all centred at origin, z-centred at z=0):
      1. MA_island_{q.name}  Metal-Air on qubit island
                             Square side = 2*pad_r, thickness = 5 nm
      2. SDA_slot_{q.name}   Substrate-Air in the annular slot
                             Square side = 2*(pad_r+gap), thickness = 3 nm
                             (island area subtracted analytically in Step 8b)
      3. MA_ground_{q.name}  Metal-Air near-qubit ground plane
                             Square side = 4*(pad_r+gap), thickness = 5 nm
                             (slot+island subtracted analytically in Step 8b)

    Returns
    -------
    dict  {interface_key: HFSS_object_name}
    """
    pad_r = q.options.pad_r       # e.g. '200um'
    gap   = q.options.pad2pk_gap  # e.g. '100um'

    t_MA  = '5um'
    t_SDA = '5um'

    boxes = {}

    # 1. MA_island
    name = f'MA_island_{q.name}'
    _draw_spr_box(design, name,
                  x0=f'-({pad_r})',  y0=f'-({pad_r})',
                  xs=f'2*({pad_r})', ys=f'2*({pad_r})',
                  thickness=t_MA)
    boxes['MA_island'] = name
    print(f'  [OK] {name:28s}  XY=2x{pad_r}  t={t_MA}')

    # 2. SDA_slot
    R_out_expr = f'({pad_r}+{gap})'
    name = f'SDA_slot_{q.name}'
    _draw_spr_box(design, name,
                  x0=f'-{R_out_expr}',  y0=f'-{R_out_expr}',
                  xs=f'2*{R_out_expr}', ys=f'2*{R_out_expr}',
                  thickness=t_SDA)
    boxes['SDA_slot'] = name
    print(f'  [OK] {name:28s}  XY=2x({pad_r}+{gap})  t={t_SDA}')

    # 3. MA_ground
    R_gnd_expr = f'2*({pad_r}+{gap})'
    name = f'MA_ground_{q.name}'
    _draw_spr_box(design, name,
                  x0=f'-{R_gnd_expr}',  y0=f'-{R_gnd_expr}',
                  xs=f'2*{R_gnd_expr}', ys=f'2*{R_gnd_expr}',
                  thickness=t_MA)
    boxes['MA_ground'] = name
    print(f'  [OK] {name:28s}  XY=4x({pad_r}+{gap})  t={t_MA}')

    return boxes


# -- Create the boxes in HFSS -------------------------------------------------
print("Creating SPR EPR boxes for CircmonG TC1...")
print()
try:
    spr_boxes = create_circmong_spr_boxes(design, circmon1)
    print()
    print("SPR boxes registered in HFSS (non-model -- solver unaffected):")
    for k, v in spr_boxes.items():
        print(f"  {k:15s} -> '{v}'")
except Exception as exc:
    import traceback; traceback.print_exc()
    spr_boxes = {
        'MA_island' : 'MA_island_TC1',
        'SDA_slot'  : 'SDA_slot_TC1',
        'MA_ground' : 'MA_ground_TC1',
    }
    print(f"\n[WARNING] Box creation failed: {exc}")
    print("Stored default names; re-run after eig_qb.sim.run() succeeds.")
    print("Box names assumed:", spr_boxes)


Creating SPR EPR boxes for CircmonG TC1...

  [OK] MA_island_TC1                 XY=2x200um  t=500nm
  [OK] SDA_slot_TC1                  XY=2x(200um+100um)  t=500nm
  [OK] MA_ground_TC1                 XY=4x(200um+100um)  t=500nm

SPR boxes registered in HFSS (non-model -- solver unaffected):
  MA_island       -> 'MA_island_TC1'
  SDA_slot        -> 'SDA_slot_TC1'
  MA_ground       -> 'MA_ground_TC1'


---
## Step 4 — Run Finite-Element Eigenmode Analysis in HFSS

HFSS replaces each Josephson Junction with its linearised inductance Lj and
solves Maxwell's eigenvalue problem for the closed cavity.  The resulting
eigenfrequency is the **bare (linearised) qubit frequency** — slightly higher
than the true dressed frequency because the cosine nonlinearity is not yet
included (that is corrected in Step 9 by pyEPR quantum analysis).

Convergence criterion: < 0.1% change in eigenfrequency between consecutive
adaptive mesh refinement passes.


In [11]:
# -- Step 4a: Trigger HFSS eigenmode solve -----------------------------------
# In Qiskit-Metal 0.5.x, eig_qb.sim.run() renders but does NOT always launch
# the HFSS solver.  Call analyze_setup() explicitly to force the solve.

print("Triggering HFSS eigenmode analysis (this may take 1-5 min)...")
_solved = False
try:
    eig_qb.sim.renderer.analyze_setup(eig_qb.sim.setup.name)
    _solved = True
    print("  HFSS solve complete via analyze_setup().")
except Exception as _e1:
    print(f"  [analyze_setup: {_e1}]")
    try:
        eig_qb.sim.renderer.pinfo.setup.analyze()
        _solved = True
        print("  HFSS solve complete via pinfo.setup.analyze().")
    except Exception as _e2:
        print(f"  [pinfo.setup.analyze: {_e2}]")
        print()
        print("  ACTION REQUIRED:")
        print("  In HFSS GUI: Simulation -> Analyze All, then re-run cells below.")

print()

# -- Step 4b: Check convergence -----------------------------------------------
try:
    eig_qb.sim.plot_convergences()
    plt.suptitle(
        "HFSS Eigenmode Convergence -- Grounded Concentric Transmon",
        fontsize=11, y=1.02)
    plt.tight_layout()
    plt.savefig('convergence.png', dpi=150, bbox_inches='tight')
    plt.show()
except Exception as e:
    print(f"  [plot_convergences: {e}]")
    print("  (No convergence data yet -- complete the HFSS solve first)")

# -- Step 4c: Report eigenfrequencies -----------------------------------------
try:
    freqs = eig_qb.get_frequencies()
    print("Eigenmode frequencies (linearised with Lj):")
    print(freqs)
    print()
    try:
        print("Convergence per pass:")
        print(eig_qb.sim.convergence_f)
    except Exception:
        pass
except Exception as e:
    print(f"  [get_frequencies: {e}]")
    print("  Ensure the HFSS solve ran successfully before proceeding to Step 5.")


INFO 03:40PM [get_setup]: 	Opened setup `Qbit_Setup`  (<class 'pyEPR.ansys.HfssEMSetup'>)
INFO 03:40PM [analyze]: Analyzing setup Qbit_Setup


Triggering HFSS eigenmode analysis (this may take 1-5 min)...


INFO 03:56PM [analyze]: Analyzing setup Qbit_Setup


  [analyze_setup: (-2147352567, 'Exception occurred.', (0, None, None, None, 0, -2147024349), None)]
  [pinfo.setup.analyze: (-2147352567, 'Exception occurred.', (0, None, None, None, 0, -2147024349), None)]

  ACTION REQUIRED:
  In HFSS GUI: Simulation -> Analyze All, then re-run cells below.

Design "Qbit_hfss_hfss" info:
	# eigenmodes    1
	# variations    0
Design "Qbit_hfss_hfss" info:
	# eigenmodes    1
	# variations    0
  [get_frequencies: No objects to concatenate]
  Ensure the HFSS solve ran successfully before proceeding to Step 5.


---
## Step 5 — Plot Electric and Magnetic Field Distributions

Expected field pattern for the **grounded concentric transmon**:

| Field | Expected distribution |
|---|---|
| **E-field** | Radially concentrated across the circular slot (Wslot gap). Strong radial E in the slot, falls off quickly inside the disk and in the metallic plane. |
| **H-field** | Circulates around the disk; peaks at the JJ bridges (east/west) where current loops close through the JJ inductance. |

A large fraction of E-field inside the substrate is expected (~90%),
consistent with typical planar transmon designs on silicon.


In [ ]:
# -- E-field and H-field plots ------------------------------------------------
# Note: eig_qb.sim.save_screenshot() takes NO arguments in QM 0.5.x

print("Plotting E-field (eigenmode 1)...")
try:
    eig_qb.sim.plot_fields('main')
    try:
        eig_qb.sim.save_screenshot()   # no filename arg in QM 0.5.x
    except Exception:
        pass   # screenshot is optional
    print("  E-field plot complete.")
except Exception as e:
    print(f"  [plot_fields E: {e}]")

print("Plotting H-field (eigenmode 1)...")
try:
    eig_qb.sim.plot_fields('main', field_type='H')
    try:
        eig_qb.sim.save_screenshot()
    except Exception:
        pass
    print("  H-field plot complete.")
except Exception as e:
    print(f"  [plot_fields H: {e}]")

try:
    eig_qb.sim.clear_fields()
except Exception:
    pass

print()
print("Interpretation checklist:")
print("  [ ] E-field peaks inside the circular slot (Wslot gap region)")
print("  [ ] E-field drops sharply outside slot and inside disk")
print("  [ ] H-field circulates around disk; peaks at JJ bridge position")
print("  [ ] No significant field at chip boundary (W x W square edge)")
print()
print("If E-field leaks to the chip boundary, increase W (design chip size).")


---
## Step 6 — EPR Analysis on the Qubit Mode

pyEPR integrates the stored electric energy over each JJ rectangle to
compute the inductive Energy Participation Ratio (EPR):

```
p_mj = (inductive energy in JJ j for mode m) / (total electric energy)
```

For the grounded concentric transmon with two symmetric JJs:
- `p_total = p_east + p_west`
- Ideally `p_total → 1` (all inductive energy in the JJs)
- Typical values: 0.85 – 0.98 for well-designed transmons

The capacitive EPR `p_cap` (small for transmon) and sign `s_mj` are also reported.


In [ ]:
# ── Configure the single junction for EPR ────────────────────────────────
# Component name must match what was used in CircmonG(design, 'TC1', ...)
comp = 'TC1'

# Remove any default placeholder
if 'jj' in eig_qb.setup.junctions:
    del eig_qb.setup.junctions['jj']

# Register the single JJ  (key used in make_jj() was 'rect_jj1')
# HFSS object names follow the convention:
#   rect : JJ_rect_Lj_{comp}_{key}
#   line : JJ_Lj_{comp}_{key}_
eig_qb.setup.junctions['rect_jj1'] = Dict(
    rect        = f'JJ_rect_Lj_{comp}_rect_jj1',
    line        = f'JJ_Lj_{comp}_rect_jj1_',
    Lj_variable = 'Lj',
    Cj_variable = 'Cj',
)

# ── Substrate dielectric for loss participation ───────────────────────────
# Qiskit-Metal names the substrate object '{chip_name}_substrate'.
# For the default chip 'main' this is 'main_substrate'.
# If your HFSS project uses a different name, replace it here.
chip_name = design.chips.main['_short_name']  # usually 'main'
substrate_obj = 'main'      # → 'main_substrate'
eig_qb.setup.dissipatives.dielectrics_bulk = [substrate_obj]

print("EPR junction configuration (single JJ):")
print(f"  Junction rect : JJ_rect_Lj_{comp}_rect_jj1")
print(f"  Junction line : JJ_Lj_{comp}_rect_jj1_")
print(f"  Substrate obj : {substrate_obj}")
print()
print(eig_qb.setup)

In [ ]:
# ── Run EPR analysis ──────────────────────────────────────────────────────
# run_epr() integrates E-field over the JJ rectangle and computes
# the inductive participation ratio p_mj and participation sign s_mj.
print("Running EPR field integration...")
eig_qb.run_epr()

# ── Access pyEPR DistributedAnalysis results ──────────────────────────────
# In this version of Qiskit-Metal, EPR data is stored on the renderer's
# epr_distributed_analysis object, NOT on eig_qb.epr_results.
da = eig_qb.sim.renderer.epr_distributed_analysis   # pyEPR DistributedAnalysis

print()
print("── EPR Participation (p_mj) ──")
print(da.results.p_mj)
print()
print("── EPR Sign (s_mj / PSR) ──")
print(da.results.sm_mj)


In [ ]:
#eig_qb.sim.close()

---
## Step 7 — Qubit Frequency and Anharmonicity

From first-order EPR perturbation theory:

| Quantity | Formula |
|---|---|
| Corrected f_01 | `f_q = f_HFSS * (1 - p_total / 2)` |
| Anharmonicity | `alpha = - p_total^2 * Ec` |
| Ec estimate | invert `f_01 = sqrt(8*Ej_eff*Ec) - Ec` |
| Ej_eff | `= 2 * (Phi0/2pi)^2 / Lj`  (two JJs in parallel) |

The precise Ec and dressed frequency are calculated in Step 9.


In [ ]:
h_planck = 6.626e-34
Phi0     = 2.068e-15

# -- HFSS bare eigenfrequency -------------------------------------------------
# Multiple fallback strategies to get f_hfss_GHz regardless of pyEPR version.
f_hfss_GHz = None

# Strategy 1: standard Qiskit-Metal wrapper
try:
    freqs_df   = eig_qb.get_frequencies()
    f_hfss_GHz = float(freqs_df.iloc[0, 0])
    print(f"  f_hfss_GHz (get_frequencies): {f_hfss_GHz:.4f} GHz")
except Exception as _e1:
    print(f"  [get_frequencies: {_e1}]")

# Strategy 2: pyEPR pinfo bare frequencies
if f_hfss_GHz is None:
    try:
        pinfo      = eig_qb.sim.renderer.pinfo
        freqs_bare = pinfo.get_freqs_bare_pd('0')
        f_hfss_GHz = float(freqs_bare.iloc[0])
        print(f"  f_hfss_GHz (pinfo bare):      {f_hfss_GHz:.4f} GHz")
    except Exception as _e2:
        print(f"  [pinfo.get_freqs_bare_pd: {_e2}]")

# Strategy 3: DistributedAnalysis results
if f_hfss_GHz is None:
    try:
        da_        = eig_qb.sim.renderer.epr_distributed_analysis
        f_hfss_GHz = float(next(iter(da_.results.freqs_hfss.values())))
        print(f"  f_hfss_GHz (da.results):      {f_hfss_GHz:.4f} GHz")
    except Exception as _e3:
        print(f"  [da.results.freqs_hfss: {_e3}]")

# Fallback: design-target placeholder
if f_hfss_GHz is None:
    f_hfss_GHz = 3.5
    print(f"  [WARNING] All frequency reads failed.")
    print(f"  Using placeholder f_hfss_GHz = {f_hfss_GHz} GHz.")
    print(f"  --> Complete HFSS solve (Step 4) and re-run from this cell.")

# -- Inductive EPR for the single junction ------------------------------------
da   = eig_qb.sim.renderer.epr_distributed_analysis
p_mj = da.results.p_mj

print()
print("DEBUG -- p_mj structure:")
print(f"  type(da.results)  : {type(da.results)}")
print(f"  type(p_mj)        : {type(p_mj)}")
try:
    print(f"  p_mj.shape        : {p_mj.shape}")
    print(f"  p_mj.index        : {list(p_mj.index)}")
    print(f"  p_mj.columns      : {list(p_mj.columns)}")
except Exception:
    pass
print(f"  p_mj:\n{p_mj}")
print()

def deep_float(v):
    if isinstance(v, (int, float)):
        return float(v)
    try:
        return float(v)
    except (TypeError, ValueError):
        pass
    if hasattr(v, 'values') and callable(v.values):
        return deep_float(next(iter(v.values())))
    if hasattr(v, '__iter__') and not isinstance(v, str):
        return deep_float(next(iter(v)))
    raise TypeError(f"Cannot extract float from {type(v)}: {v!r}")

import pandas as pd

def get_junction_participation(p_mj, jj_key, mode=0):
    for _strat, _fn in [
        ("A iloc[mode][key]",    lambda: p_mj.iloc[mode][jj_key]),
        ("B iloc[mode,0]",       lambda: p_mj.iloc[mode, 0]),
        ("C loc[key].iloc[0]",   lambda: p_mj.loc[jj_key].iloc[mode]),
        ("D nested DataFrame",   lambda: p_mj.iloc[0].iloc[mode][jj_key]
                                         if isinstance(p_mj.iloc[0], pd.DataFrame) else None),
    ]:
        try:
            raw = _fn()
            if raw is not None:
                return deep_float(raw)
        except Exception:
            pass
    raise RuntimeError(f"Could not extract p_mj for '{jj_key}'.")

try:
    p_jj1 = get_junction_participation(p_mj, 'rect_jj1', mode=0)
    print(f"  Extracted p_jj1 = {p_jj1:.4f}")
except RuntimeError as e:
    print(f"  ERROR: {e}")
    p_jj1 = 0.95    # typical value; replace after successful EPR run
    print(f"  Using fallback p_jj1 = {p_jj1}")

p_total = p_jj1

print("=" * 58)
print(f"  HFSS bare eigenfrequency  : {f_hfss_GHz:.4f} GHz")
print(f"  p_inductive rect_jj1      : {p_jj1:.4f}")
print("=" * 58)

f_q_GHz    = f_hfss_GHz * (1.0 - p_total / 2.0)
print(f"\n  Corrected f_01 (EPR O1)   : {f_q_GHz:.4f} GHz")

Lj_nH      = float(eig_qb.sim.setup.vars.Lj.replace(' nH', '').strip())
Ej_GHz     = (Phi0 / (2 * np.pi))**2 / (Lj_nH * 1e-9) / (h_planck * 1e9)
Ej_eff_GHz = Ej_GHz

print(f"\n  Lj (single JJ)            : {Lj_nH:.1f} nH")
print(f"  Ej                        : {Ej_GHz:.3f} GHz")

def f01_eq(Ec, Ej, target):
    return np.sqrt(8 * Ej * Ec) - Ec - target

try:
    from scipy.optimize import brentq
    Ec_GHz     = brentq(f01_eq, 0.01, 1.0, args=(Ej_eff_GHz, f_hfss_GHz))
    alpha_GHz  = -(p_total**2) * Ec_GHz
    EjEc_ratio = Ej_eff_GHz / Ec_GHz
    print(f"\n  Estimated Ec              : {Ec_GHz*1e3:.1f} MHz")
    print(f"  Estimated anharmonicity   : {alpha_GHz*1e3:.1f} MHz")
    print(f"  Estimated Ej/Ec           : {EjEc_ratio:.1f}")
    ok_f  = "PASS" if 3.0 <= f_hfss_GHz <= 4.8 else "FAIL -- adjust Lj"
    ok_ej = "PASS" if EjEc_ratio > 50            else "FAIL -- reduce Lj"
    print(f"\n  Target f_01 in 3-4.8 GHz : {f_hfss_GHz:.4f} GHz  [{ok_f}]")
    print(f"  Target Ej/Ec > 50        : {EjEc_ratio:.1f}       [{ok_ej}]")
except Exception as ex:
    print(f"  Could not estimate Ec: {ex}")
    Ec_GHz     = 0.140
    EjEc_ratio = Ej_eff_GHz / Ec_GHz


---
## Step 8 -- Loss Analysis: Bulk EPR, SPR, and T1 Budget

### 8a  Bulk substrate dielectric participation (automatic)
pyEPR reports this directly after `run_epr()`:
```
1/T1_bulk = p_sub x omega_01 x tan_delta_Si
```
For high-purity float-zone Si at mK: tan_delta ~ 3e-6 -> bulk T1 >> 1 ms (not limiting).

---

### 8b  Surface Participation Ratio (SPR) -- theory

Thin nm-scale interfacial dielectric layers at each metal/insulator boundary
carry anomalously high loss (tan_delta ~ 1e-3). Their energy participation is:

```
p_s = (eps_s * eps_0 * t_s) / (2 * U_E) * integral_A |E_t(r)|^2 dA
```

where **E_t** is the tangential electric field at the interface, t_s is the
loss-layer thickness, and U_E is the total electric energy in the eigenmode.

T1 contribution from surface s:
```
1/T1_s = p_s * omega_01 * tan_delta_s
```

#### Interface parameters for Al / Si system

| Interface | Symbol | Location | t [nm] | eps_r | tan_delta |
|---|---|---|---|---|---|
| Metal-Air | MA | Top of Al film (AlOx) | 3 | 10 | 1e-3 |
| Substrate-Air | SDA | Exposed Si in slot | 2 | 11.4 | 1e-3 |
| Metal-Substrate | MS | Bottom of Al film | 2 | 11.4 | 3e-4 |

References: Mueller et al. PRAppl 2019; Martinis & Megrant arXiv:1410.5793;
Wenner et al. APL 2011.

#### CircmonG geometry for SPR (pad_r=200 um, pad2pk_gap=100 um)

| Region | Area (circular) | Dominant interface |
|---|---|---|
| Island (r < 200 um) | pi*(200)^2 ~ 0.126 mm^2 | MA |
| Slot (200 < r < 300 um) | pi*(300^2-200^2) ~ 0.157 mm^2 | SDA |
| Ground plane (r > 300 um) | chip - slot - island | MA, weak E-field |

#### EPR-box approach
A thin non-model rectangular box at z = 0 serves as integration domain.
HFSS field calculator integrates |E_t|^2 over the box volume; for
thickness t_box << qubit scale:

```
integral_vol |E_t|^2 dV  ~  t_box * integral_surf |E_t|^2 dA
```

Area correction factors account for the square-vs-circular geometry.


In [ ]:
# -- Guard: ensure f_hfss_GHz is always defined (cascade from cell 24) -------
if 'f_hfss_GHz' not in dir() or f_hfss_GHz is None:
    f_hfss_GHz = 3.5
    print(f"  [WARNING] f_hfss_GHz not defined; using {f_hfss_GHz} GHz placeholder.")
    print(f"  Complete HFSS solve (Step 4) and re-run cell 24 first.")

# ── 8a: Bulk substrate EPR ────────────────────────────────────────────────
print("=== 8a: Bulk Substrate EPR ===")
print()

# EPRanalysis stores the electric energies as attributes after run_epr():
#   eig_qb.energy_elec      — total electric energy in the simulation volume [J]
#   eig_qb.energy_elec_sub  — electric energy inside the substrate dielectric [J]
# Substrate participation = energy_elec_sub / energy_elec
try:
    p_sub = float(eig_qb.energy_elec_sub) / float(eig_qb.energy_elec)
    print(f"  p_substrate (bulk Si)     : {p_sub:.4f}  ({p_sub*100:.1f}%)")
    print(f"  (energy_elec_sub / energy_elec)")
except (AttributeError, ZeroDivisionError, TypeError) as e:
    # Fallback: pyEPR also prints 'EPR of substrate = XX%' in the run_epr() output.
    # If the attribute isn't available, read the printed value above.
    p_sub = 0.92   # typical planar transmon on Si (replace with printed value)
    print(f"  [energy_elec_sub not available ({e})]")
    print(f"  Using fallback estimate    : p_sub ~ {p_sub}")
    print(f"  (check the run_epr() output above for 'EPR of substrate')")

# ── Dielectric-limited T1 ─────────────────────────────────────────────────
tan_delta_Si = 3e-6          # bulk Si at mK (high-purity crystalline)
omega_q      = 2 * np.pi * f_hfss_GHz * 1e9
Q_diel       = 1.0 / (p_sub * tan_delta_Si)
T1_diel_us   = Q_diel / omega_q * 1e6

print(f"\n  tan(delta) Si assumed     : {tan_delta_Si:.0e}")
print(f"  Dielectric Q              : {Q_diel:.2e}")
print(f"  Dielectric-limited T1     : {T1_diel_us:.0f} us")


In [ ]:
# -- 8b.1: Interface parameters and geometry correction factors ---------------
import numpy as np

eps_0 = 8.854e-12      # F/m

# -- Parse qubit geometry -----------------------------------------------------
pad_r_um  = float(circmon1.options.pad_r.replace('um',''))
gap_um    = float(circmon1.options.pad2pk_gap.replace('um',''))
R_in_um   = pad_r_um
R_out_um  = pad_r_um + gap_um

A_island_circ = np.pi * R_in_um**2                  # um^2
A_slot_ann    = np.pi * (R_out_um**2 - R_in_um**2)  # um^2
A_island_box  = (2 * R_in_um)**2
A_slot_box    = (2 * R_out_um)**2
A_gnd_box     = (4 * R_out_um)**2
A_gnd_net     = A_gnd_box - A_slot_box               # ground ring (approx)

# Area correction: ratio of actual circular/annular area to bounding square
f_area = {
    'MA_island' : A_island_circ / A_island_box,  # pi/4 ~ 0.785
    'SDA_slot'  : A_slot_ann    / A_slot_box,    # ~ 0.436
    'MA_ground' : A_gnd_net     / A_gnd_box,     # ring fraction
    'MS_island' : A_island_circ / A_island_box,  # same as MA_island
}

print("=== 8b.1: CircmonG Geometry & Area Correction Factors ===")
print(f"  pad_r         : {pad_r_um:.0f} um")
print(f"  pad2pk_gap    : {gap_um:.0f} um")
print(f"  Island area   : {A_island_circ:.0f} um^2  ({A_island_circ*1e-6:.4f} mm^2)")
print(f"  Slot area     : {A_slot_ann:.0f} um^2  ({A_slot_ann*1e-6:.4f} mm^2)")
print()
print("  Area correction factors (circular area / bounding square):")
for k, f in f_area.items():
    print(f"    {k:15s}: {f:.4f}")
print()

# -- Literature interface parameters ------------------------------------------
# Update tan_d from your specific material stack if known
surfaces = {
    'MA_island': dict(
        eps_r=10.0, t_m=3e-9, t_box=5e-9, tan_d=1e-3,
        label='Metal-Air (island)'),
    'SDA_slot': dict(
        eps_r=11.4, t_m=2e-9, t_box=3e-9, tan_d=1e-3,
        label='Substrate-Air (slot)'),
    'MA_ground': dict(
        eps_r=10.0, t_m=3e-9, t_box=5e-9, tan_d=1e-3,
        label='Metal-Air (ground, near-field)'),
    'MS_island': dict(
        eps_r=11.4, t_m=2e-9, t_box=3e-9, tan_d=3e-4,
        label='Metal-Substrate (island)'),
}
print("  Interface parameters loaded (4 interfaces).")


In [ ]:
# -- 8b.2: HFSS Field Calculator -- surface integrals integral|E_t|^2 dV -----
#
# Uses HFSS COM interface (via pyEPR pinfo) to integrate |E_t|^2 over each
# SPR box volume.  For a thin box of thickness t_box at z=0:
#   integral_box |E_t|^2 dV  ~  t_box * integral_surf |E_t|^2 dA
# |E_t|^2 = Mag_E^2 - ScalarZ(E)^2   (z-normal planar surface)
#
# Prerequisite: HFSS eigenmode solve converged (Step 4).

print("=== 8b.2: HFSS Field Calculator -- SPR surface integrals ===")
print()

# Guard
if 'f_hfss_GHz' not in dir() or f_hfss_GHz is None:
    f_hfss_GHz = 3.5
    print(f"  [WARNING] f_hfss_GHz defaulting to {f_hfss_GHz} GHz.")

# -- Access pinfo and field calculator ----------------------------------------
renderer  = eig_qb.sim.renderer
pinfo     = renderer.pinfo
oDesign_h = pinfo.design._design
oFields   = oDesign_h.GetModule('FieldsReporter')

# -- Total electric energy: robust extraction (may be addict.Dict in pyEPR) ---
def _extract_energy(val):
    # Safely extract a float from an energy attribute (may be addict.Dict).
    if isinstance(val, (int, float)):
        return float(val)
    try:
        return float(val)
    except (TypeError, ValueError):
        pass
    # addict.Dict / plain dict: iterate values
    if hasattr(val, 'items'):
        for _, v in val.items():
            try:
                return _extract_energy(v)
            except Exception:
                pass
    # numpy array or other iterable
    if hasattr(val, '__iter__') and not isinstance(val, str):
        for v in val:
            try:
                return _extract_energy(v)
            except Exception:
                pass
    raise TypeError(f"Cannot extract energy float from {type(val)}: {val!r}")

U_E = None
for _src, _getter in [
    ("eig_qb.energy_elec",           lambda: eig_qb.energy_elec),
    ("da.results.energy_elec",        lambda: eig_qb.sim.renderer.epr_distributed_analysis.results.energy_elec),
]:
    try:
        U_E = _extract_energy(_getter())
        print(f"  U_E ({_src}) : {U_E:.4e} J")
        break
    except Exception as _exc:
        print(f"  [{_src} failed: {_exc}]")

if U_E is None or U_E == 0:
    hbar = 1.055e-34
    U_E  = 0.5 * hbar * 2 * np.pi * f_hfss_GHz * 1e9
    print(f"  U_E (1/2-photon estimate) : {U_E:.4e} J  <-- replace with HFSS value")
print()


def hfss_E_tan_sq_vol(oFields, vol_name, variation='0', freq_ghz=3.5):
    # Integrate |E_t|^2 over a named volume via HFSS field calculator.
    # |E_t|^2 = Mag_E^2 - ScalarZ(E)^2  (z-normal, chip at z=0)
    freq_str = f'{freq_ghz:.6f}GHz'
    oFields.CalcStack('clear')
    oFields.EnterQty('Mag_E')
    oFields.CalcOp('Sqr')
    oFields.EnterQty('E')
    oFields.CalcOp('ScalarZ')
    oFields.CalcOp('Mag')
    oFields.CalcOp('Sqr')
    oFields.CalcOp('-')
    oFields.EnterVol(vol_name)
    oFields.CalcOp('Integrate')
    # Try two call signatures for HFSS version compatibility
    try:
        result = oFields.GetTopEntryValue(f'Phase=`0deg`', variation)
        return float(result[0])
    except Exception:
        result = oFields.GetTopEntryValue(variation, freq_str)
        return float(result[0])


# -- Run field calculator for each SPR box ------------------------------------
variation = '0'
E_tan_sq_vol_results = {}

print(f"  {'Box':28s}  {'integral|E_t|^2 dV [V^2 m]':>26s}  Status")
print(f"  {'-'*28}  {'-'*26}  {'-'*8}")

for key, sp in surfaces.items():
    box_name = spr_boxes.get(key, f'{key}_TC1')
    try:
        val = hfss_E_tan_sq_vol(oFields, box_name, variation, f_hfss_GHz)
        E_tan_sq_vol_results[key] = val
        status = 'OK'
    except Exception as e:
        E_tan_sq_vol_results[key] = None
        status = f'FAILED: {str(e)[:35]}'
    print(f"  {box_name:28s}  {str(E_tan_sq_vol_results[key]):>26s}  {status}")

print()
n_ok = sum(1 for v in E_tan_sq_vol_results.values() if v is not None)
print(f"  Field calculator: {n_ok}/{len(surfaces)} interfaces succeeded.")
if n_ok < len(surfaces):
    print("  Failed -> proxy estimates used in 8b.3.")
    print("  Common causes: HFSS not connected, box name mismatch, no solution.")
    print("  List HFSS objects: design.renderers.hfss.modeler._modeler.GetChildNames()")


In [ ]:
# Guard: f_hfss_GHz must be defined (cascade from cell 24)
if 'f_hfss_GHz' not in dir() or f_hfss_GHz is None:
    f_hfss_GHz = 3.5
    print(f"  [WARNING] f_hfss_GHz defaulting to {f_hfss_GHz} GHz -- re-run cell 24.")
# Guard: E_tan_sq_vol_results must exist (cascade from cell 28)
if 'E_tan_sq_vol_results' not in dir():
    E_tan_sq_vol_results = {k: None for k in surfaces}
    print("  [WARNING] E_tan_sq_vol_results not found -- using proxy estimates.")

# -- 8b.3: Compute SPR, Q-factors, and T1 budget ------------------------------
#
# SPR formula:
#   p_s = (eps_s * eps_0 * t_s) / (2 * U_E) * integral|E_t|^2 dA
#
# Thin-box volume integral:
#   integral|E_t|^2 dA ~ (integral|E_t|^2 dV / t_box) * f_area_correction
#
# T1 contribution:   1/T1_s = p_s * omega_01 * tan_delta_s

print("=== 8b.3: SPR Values, Q-factors, and T1 Budget ===")
print()

omega_q_rad = 2 * np.pi * f_hfss_GHz * 1e9   # rad/s

spr_results = {}

for key, sp in surfaces.items():
    fc_val = E_tan_sq_vol_results.get(key)   # integral|E_t|^2 dV [V^2 m] or None
    f_ac   = f_area.get(key, 1.0)

    if fc_val is not None and fc_val > 0:
        # Method A: HFSS field calculator
        # integral|E_t|^2 dA = (integral|E_t|^2 dV / t_box) * f_area_correction
        E_tan_sq_surf = (fc_val / sp['t_box']) * f_ac   # V^2/m
        p_s    = sp['eps_r'] * eps_0 * sp['t_m'] * E_tan_sq_surf / (2 * U_E)
        method = 'HFSS field calc'
    else:
        # Method B: proxy from bulk substrate participation (order of magnitude)
        t_Si_m = 280e-6   # substrate height (280 um chip z-size)
        p_s    = p_sub * (sp['eps_r'] * sp['t_m']) / (11.4 * t_Si_m) * f_ac
        E_tan_sq_surf = 2 * U_E * p_s / (sp['eps_r'] * eps_0 * sp['t_m'])
        method = 'proxy (HFSS calc failed)'

    Q_s  = 1.0 / (p_s * sp['tan_d']) if p_s > 0 else float('inf')
    T1_s = Q_s / omega_q_rad * 1e6   # us

    spr_results[key] = dict(p_s=p_s, Q_s=Q_s, T1_us=T1_s,
                             E_tan_sq_surf=E_tan_sq_surf, method=method)

# Print SPR table
print(f"  {'Interface':35s}  {'p_s':>10s}  {'Q_surf':>10s}  {'T1 [us]':>10s}  Method")
print(f"  {'-'*35}  {'-'*10}  {'-'*10}  {'-'*10}  {'-'*22}")
for key, res in spr_results.items():
    sp   = surfaces[key]
    flag = ' *' if 'proxy' in res['method'] else ''
    print(f"  {sp['label']:35s}  {res['p_s']:10.3e}  {res['Q_s']:10.2e}"
          f"  {res['T1_us']:10.1f}  {res['method'][:22]}{flag}")
print()
print("  * = proxy estimate; run HFSS field calculator for accurate values")
print()

# T1 budget
print("-" * 70)
print("  T1 BUDGET  (all contributions)")
print("-" * 70)
print(f"  {'Loss channel':35s}  {'p_s':>10s}  {'tan d':>8s}  {'T1 [us]':>10s}")
print(f"  {'-'*35}  {'-'*10}  {'-'*8}  {'-'*10}")

rate_sum = 0.0
for key, res in spr_results.items():
    sp   = surfaces[key]
    rate = 1.0 / res['T1_us'] if res['T1_us'] > 0 else 0.0
    rate_sum += rate
    print(f"  {sp['label']:35s}  {res['p_s']:10.3e}  {sp['tan_d']:8.0e}  {res['T1_us']:10.1f}")

rate_bulk = 1.0 / T1_diel_us
rate_sum += rate_bulk
print(f"  {'Bulk substrate (Si)':35s}  {p_sub:10.4f}  {tan_delta_Si:8.0e}  {T1_diel_us:10.1f}")
print(f"  {'-'*35}  {'-'*10}  {'-'*8}  {'-'*10}")

T1_total_us = 1.0 / rate_sum if rate_sum > 0 else float('inf')
print(f"  {'Total (all channels)':35s}  {'--':>10s}  {'--':>8s}  {T1_total_us:10.1f}")
print("-" * 70)
print()
print(f"  Predicted T1 (surface + bulk) : {T1_total_us:.0f} us  ({T1_total_us/1000:.2f} ms)")

rate_by_channel = {k: 1.0/v['T1_us'] for k, v in spr_results.items() if v['T1_us']>0}
rate_by_channel['bulk_Si'] = rate_bulk
dominant_key = max(rate_by_channel, key=rate_by_channel.get)
dominant_T1  = 1.0 / rate_by_channel[dominant_key]
print(f"  Dominant loss channel         : {dominant_key}  (T1 = {dominant_T1:.0f} us)")


In [ ]:
# Guard
if 'spr_results' not in dir():
    print("[WARNING] spr_results not defined -- run cell 29 (8b.3) first.")
    spr_results = {}
if 'T1_total_us' not in dir():
    T1_total_us = float('inf')

# -- 8b.4: SPR Quality Assessment and Improvement Targets --------------------
#
# State-of-the-art targets (2022-2024, Al on Si, planar transmon):
#   p_MA  < 3e-6  -> T1_MA  > 500 us  (Kim+ Nature 2021; Ganjam+ 2024)
#   p_SDA < 2e-6  -> T1_SDA > 700 us
#   p_MS  < 1e-6  -> T1_MS  > 2 ms    (less critical, lower tan_d)

print("=== 8b.4: SPR Quality Assessment ===")
print()

spr_targets = {
    'MA_island' : dict(p_max=3.0e-6, T1_min=500,  grade='state-of-art'),
    'SDA_slot'  : dict(p_max=2.0e-6, T1_min=700,  grade='state-of-art'),
    'MA_ground' : dict(p_max=1.5e-6, T1_min=1000, grade='state-of-art'),
    'MS_island' : dict(p_max=1.0e-6, T1_min=2000, grade='state-of-art'),
}

print(f"  {'Interface':35s}  {'p_s':>10s}  {'Target':>10s}  "
      f"{'Margin':>8s}  {'T1 [us]':>10s}  Status")
print(f"  {'-'*35}  {'-'*10}  {'-'*10}  {'-'*8}  {'-'*10}  {'-'*8}")

needs_improvement = []
for key, tgt in spr_targets.items():
    if key not in spr_results:
        continue
    res    = spr_results[key]
    p_s    = res['p_s']
    margin = tgt['p_max'] / p_s    # >1 = good, <1 = needs improvement
    status = 'PASS' if margin >= 1 else 'IMPROVE'
    if margin < 1:
        needs_improvement.append((key, margin, p_s, tgt['p_max']))
    print(f"  {surfaces[key]['label']:35s}  {p_s:10.3e}  {tgt['p_max']:10.3e}  "
          f"{margin:8.2f}x  {res['T1_us']:10.1f}  {status}")

print()
T1_target_us = 500
ok_T1 = 'PASS' if T1_total_us >= T1_target_us else 'IMPROVE'
print(f"  Predicted T1_total  : {T1_total_us:.0f} us")
print(f"  Target T1 > {T1_target_us} us   : {ok_T1}")
print()

if not needs_improvement:
    print("  All SPR values within state-of-art targets.")
    print("  CircmonG is surface-loss competitive.")
else:
    print("  Channels requiring improvement:")
    for key, margin, p_val, p_tgt in needs_improvement:
        factor = p_val / p_tgt
        print(f"    {key}: p_s={p_val:.2e}  target={p_tgt:.2e}  --> need {factor:.1f}x reduction")
    print()
    p_MA  = spr_results.get('MA_island',  {}).get('p_s', 0)
    p_SDA = spr_results.get('SDA_slot',   {}).get('p_s', 0)
    if p_MA >= p_SDA:
        print(f"  Primary: Metal-Air (p_MA={p_MA:.2e})")
        print(f"  -> High tangential E-field on island metal surface")
        print(f"  -> Remedy: increase pad_r to spread field over larger area (see 8b.5)")
    else:
        print(f"  Primary: Substrate-Air in slot (p_SDA={p_SDA:.2e})")
        print(f"  -> High E-field at exposed Si in the annular slot")
        print(f"  -> Remedy: increase pad2pk_gap to dilute field in slot (see 8b.5)")

print()
print("  See Step 8b.5 for parametric scaling study and improvement strategies.")


In [ ]:
# Guards
if 'spr_results' not in dir():
    spr_results = {}
    print("[WARNING] spr_results not defined -- run cells 27-29 first.")
if 'omega_q_rad' not in dir():
    omega_q_rad = 2 * np.pi * f_hfss_GHz * 1e9

# -- 8b.5: Parametric SPR Scaling Study ---------------------------------------
#
# Analytical scaling estimates (parallel-plate + fringing-field limits):
#   p_MA  ~ (R_ref / R)^2       larger island dilutes E^2 over area
#   p_SDA ~ sqrt(G_ref / G)     wider gap lowers field peak in slot
#
# These are ORDER-OF-MAGNITUDE estimates only.
# HFSS simulation is the authoritative source for accurate SPR values.

print("=== 8b.5: Parametric SPR Scaling Study ===")
print()
print("  Analytical scaling (approximate) -- verify with HFSS for each geometry")
print()

R_ref_um  = pad_r_um
G_ref_um  = gap_um
p_MA_ref  = spr_results.get('MA_island', {}).get('p_s', 3e-6)
p_SDA_ref = spr_results.get('SDA_slot',  {}).get('p_s', 2e-6)

try:
    Ec_ref_MHz = Ec_ND * 1e3
except NameError:
    Ec_ref_MHz = 140.0   # design target fallback
try:
    Ej_ref_GHz = Ej_eff_qa
except NameError:
    Ej_ref_GHz = Ej_eff_GHz

# Table 1: p_MA vs pad_r
pad_r_sweep = np.array([150, 175, 200, 225, 250, 275, 300])  # um
print(f"  Table 1: pad_r sweep (gap fixed at {G_ref_um:.0f} um)")
print(f"  {'pad_r [um]':>10s}  {'p_MA':>10s}  {'T1_MA [us]':>12s}  "
      f"{'Ec [MHz]':>10s}  {'Ej/Ec':>7s}  Note")
print(f"  {'-'*10}  {'-'*10}  {'-'*12}  {'-'*10}  {'-'*7}  {'-'*10}")

tan_d_MA = surfaces['MA_island']['tan_d']
for R in pad_r_sweep:
    p_sc    = p_MA_ref  * (R_ref_um / R)**2           # p ~ 1/R^2
    T1_sc   = (1.0 / (p_sc * tan_d_MA)) / omega_q_rad * 1e6
    Ec_sc   = Ec_ref_MHz * (R_ref_um / R)             # Ec ~ 1/R
    EjEc_sc = Ej_ref_GHz * 1e3 / Ec_sc
    note    = '<- current' if R == R_ref_um else ''
    print(f"  {R:>10.0f}  {p_sc:10.3e}  {T1_sc:12.0f}  {Ec_sc:10.1f}  {EjEc_sc:7.0f}  {note}")

print()

# Table 2: p_SDA vs pad2pk_gap
gap_sweep = np.array([50, 75, 100, 125, 150, 175, 200])  # um
print(f"  Table 2: gap sweep (pad_r fixed at {R_ref_um:.0f} um)")
print(f"  {'gap [um]':>9s}  {'p_SDA':>10s}  {'T1_SDA [us]':>13s}  "
      f"{'p_SDA/p_ref':>12s}  Note")
print(f"  {'-'*9}  {'-'*10}  {'-'*13}  {'-'*12}  {'-'*10}")

tan_d_SDA = surfaces['SDA_slot']['tan_d']
for G in gap_sweep:
    p_sc  = p_SDA_ref * np.sqrt(G_ref_um / G)    # p ~ 1/sqrt(G)
    T1_sc = (1.0 / (p_sc * tan_d_SDA)) / omega_q_rad * 1e6
    ratio = p_sc / p_SDA_ref
    note  = '<- current' if G == G_ref_um else ''
    print(f"  {G:>9.0f}  {p_sc:10.3e}  {T1_sc:13.0f}  {ratio:12.2f}  {note}")

print()
print("  Scaling notes:")
print("  - p_MA  ~ 1/R^2  : larger island dilutes field density on metal surface")
print("  - p_SDA ~ 1/sqrt(G): wider slot lowers field peak (fringing-field regime)")
print("  - Larger pad_r -> lower Ec -> retune Lj to maintain f_01 target")
print("  - Recommendation: pad_r=260-280 um, gap=130-150 um for T1 > 1 ms target")


---
### 8b.6 -- SPR Improvement Roadmap for CircmonG

#### Geometric changes (no fab change required)

| Action | Effect on SPR | Effect on Hamiltonian | Priority |
|---|---|---|---|
| pad_r: 200 -> 270 um | p_MA down ~40% | Ec down ~26% (re-tune Lj) | High |
| pad2pk_gap: 100 -> 140 um | p_SDA down ~18% | Ec down ~5% | High |
| Reduce chip size W | p_MA_ground down | Isolation reduced | Medium |
| Rounded slot corners | p_SDA hotspot reduced | None | Low |

#### Process improvements (require fab change)

| Improvement | Target interface | Expected T1 gain |
|---|---|---|
| Increase deposition vacuum (< 1e-9 Torr) | MA | 2-5x |
| In-situ Ar+ ion milling before Al deposition | MA + MS | 2-4x |
| 2% HF dip (30 s) + immediate vacuum load | SDA | 3-10x |
| H-terminated Si passivation after HF etch | SDA | 3-10x |
| Switch to TiN or NbTiN superconductor | MA | 2-3x (lower tan_d) |
| Epitaxial Al on InAs substrate | MA | 5-20x (no native oxide) |

#### Simulation improvements

1. **Mesh refinement at metal edges**: add length-based mesh operation,
   max edge length 1 um within 5 um of all metal boundaries.
   This improves accuracy of the E-field gradient at slot edges.

2. **Explicit thin dielectric layers in HFSS**: model the 3 nm AlOx as a
   physical material layer (eps_r=10, tan_d=1e-3) in the geometry.
   Re-run eigenmode with dielectric loss enabled in EPR setup.
   This directly extracts oxide energy without field-calculator approximation.

3. **Tighten convergence**: set max_delta_f=0.05% and min_converged=3.
   Surface field energy near edges is sensitive to mesh convergence.

4. **Validate with field plots**: after solving, plot |E_t| on MA and SDA
   surfaces to confirm that integration captures the physical hotspots.
   For CircmonG, peak |E_t| occurs at the inner/outer edges of the slot.

#### Expected performance after geometric + process improvements

With pad_r=270 um, gap=140 um, float-zone Si (HF cleaned), Al at 1e-9 Torr:
- p_MA  ~ 1-2e-6  -> T1_MA  ~ 700-1400 us
- p_SDA ~ 0.5-1e-6 -> T1_SDA ~ 1400-2800 us
- p_bulk ~ 0.9    -> T1_bulk > 10 ms  (not limiting)
- **Predicted T1 ~ 500-1000 us** -- consistent with state-of-art (2024)

References:
- Kim et al. Nature 2021 (T1~300 us, optimized surface treatment)
- Ganjam et al. Nature Commun. 2024 (T1>1 ms, TiN on sapphire)
- Place et al. Nature Commun. 2021 (T1~300 us, careful Si passivation)
- Wang et al. PRX Quantum 2022 (T1 vs p_MA correlation study)


---
## Step 9 — Quantum Analysis: Ej, Ec, Ej/Ec and PSR

`analyze_all_variations()` reconstructs the full Josephson Hamiltonian
from the EPR participation ratios, using:
- Non-degenerate perturbation theory (ND): most accurate
- First-order perturbation (O1): faster cross-check

**Key outputs**:
| Symbol | Source in results dict | Units |
|---|---|---|
| f_01 | `results['f_ND'][0]` | GHz |
| Ec (= -alpha) | `-results['chi_ND'][0,0]` | GHz |
| alpha | `results['chi_ND'][0,0]` | GHz |
| Ej per JJ | `get_Ejs()` | GHz |
| Ej/Ec | `Ej_eff / Ec` | — |
| PSR (sign) | `epr_results['sign_mj']` | ±1 |


In [ ]:
# ── Run full quantum analysis ─────────────────────────────────────────────
# FIX: analyze_all_variations() lives on pyEPR's QuantumAnalysis class, NOT
# on Qiskit-Metal's EPRanalysis wrapper (calling eig_qb.analyze_all_variations()
# raises AttributeError).  We instantiate QuantumAnalysis directly from the
# HDF5 file that run_epr() saved in Step 6, then call it on that object.
#
# SECOND FIX: da.data_filename may carry a stale timestamp — pyEPR records
# the time when run_epr() was *called*, but the .npz on disk is stamped when
# HFSS *finished* writing it (potentially minutes earlier).  Instead of
# trusting the filename directly, we scan the same directory for the real
# data file (the .npz that is NOT the HamiltonianResultsContainer).

import pyEPR as epr
import pathlib

da = eig_qb.sim.renderer.epr_distributed_analysis

# ── Locate the actual .npz data file on disk ─────────────────────────────
data_dir = pathlib.Path(da.data_filename).parent

npz_candidates = [
    f for f in data_dir.glob('*.npz')
    if 'HamiltonianResultsContainer' not in f.name
]

if not npz_candidates:
    raise FileNotFoundError(
        f"No pyEPR data .npz found in: {data_dir}\n"
        f"Expected by da.data_filename: {da.data_filename}"
    )

# Pick the most recently modified one (handles multiple runs in same folder)
actual_data_file = max(npz_candidates, key=lambda f: f.stat().st_mtime)

print(f"da.data_filename (may be stale) : {da.data_filename}")
print(f"Actual .npz found on disk       : {actual_data_file}")
print()

# ── Instantiate QuantumAnalysis from the real file ───────────────────────
qa = epr.QuantumAnalysis(str(actual_data_file))

# ── Run full quantum analysis ─────────────────────────────────────────────
print("Running analyze_all_variations()...")
print("  cos_trunc  = 8  — Taylor expansion of cos(phi) to 8th order")
print("  fock_trunc = 7  — Fock space size: levels |0> to |6>")
print()

qa.analyze_all_variations(
    cos_trunc  = 8,
    fock_trunc = 7,
)

print("Quantum analysis complete.")


In [ ]:
# ── Extract Ej, Ec, Ej/Ec ────────────────────────────────────────────────
variation = '0'

print("=" * 62)
print("  QUANTUM HAMILTONIAN PARAMETERS")
print("=" * 62)

# ── Josephson energy (single junction) ───────────────────────────────────
try:
    Ejs = qa.get_Ejs(variation=variation)    # was: eig_qb.get_Ejs()
    print("\n  Josephson energy Ej (GHz):")
    for jname, jval in Ejs.items():
        print(f"    {jname:20s}: {float(jval):.4f} GHz  ({float(jval)*1e3:.1f} MHz)")
    # Single JJ: Ej_eff = Ej (the one junction value)
    Ej_eff_qa = deep_float(list(Ejs.items())[0][1])
    print(f"    {'Ej (single JJ)':20s}: {Ej_eff_qa:.4f} GHz")
except Exception as ex:
    print(f"  [Ejs: {ex}]")
    Ej_eff_qa = Ej_eff_GHz

# ── Chi matrix → Ec, anharmonicity ───────────────────────────────────────
try:
    res      = qa.results[variation]         # was: eig_qb.results[variation]
    f_ND     = res['f_ND']
    f_O1     = res['f_O1']
    chi_ND   = res['chi_ND']

    Ec_ND    = -float(chi_ND[0, 0])
    alpha_ND =  float(chi_ND[0, 0])
    EjEc     = Ej_eff_qa / Ec_ND

    print(f"\n  Dressed qubit frequency:")
    print(f"    f_ND  (pert. theory)  : {float(f_ND[0]):.6f} GHz")
    print(f"    f_O1  (1st order)     : {float(f_O1[0]):.6f} GHz")

    print(f"\n  Charging energy  Ec   : {Ec_ND*1e3:.2f} MHz")
    print(f"  Anharmonicity  alpha  : {alpha_ND*1e3:.2f} MHz")

    print(f"\n  ─── Ej/Ec (single JJ) ───")
    print(f"    Ej                  : {Ej_eff_qa:.4f} GHz = {Ej_eff_qa*1e3:.1f} MHz")
    print(f"    Ec                  : {Ec_ND*1e3:.2f} MHz")
    print(f"    Ej/Ec               : {EjEc:.1f}")

    ok_f  = "PASS" if 3.0 <= float(f_ND[0]) <= 4.8 else "FAIL"
    ok_ec = "PASS" if 120  <= Ec_ND*1e3    <= 160   else "FAIL"
    ok_ej = "PASS" if EjEc > 50                     else "FAIL"

    print(f"\n  ─── Target check ───")
    print(f"    f_01 in 3-4.8 GHz   : {float(f_ND[0]):.4f} GHz  [{ok_f}]")
    print(f"    Ec  in 120-160 MHz  : {Ec_ND*1e3:.1f} MHz  [{ok_ec}]")
    print(f"    Ej/Ec > 50          : {EjEc:.1f}         [{ok_ej}]")

    if ok_f == "FAIL":
        delta = float(f_ND[0]) - 3.8   # midpoint
        print(f"    --> {'Increase' if delta < 0 else 'Decrease'} Lj to "
              f"{'raise' if delta < 0 else 'lower'} f_01")
    if ok_ec == "FAIL":
        print("    --> Adjust pad_r or pad2pk_gap to change Ec")

except Exception as ex:
    print(f"  [chi_ND not available: {ex}]")


In [ ]:
# ── PSR: Participation Sign Ratio ────────────────────────────────────────
print("=" * 62)
print("  PARTICIPATION SIGN RATIO (PSR  /  s_mj)")
print("=" * 62)
print()

try:
    da    = eig_qb.sim.renderer.epr_distributed_analysis
    sm_mj = da.results.sm_mj

    print("DEBUG — sm_mj structure:")
    print(f"  type(sm_mj) : {type(sm_mj)}")
    print(f"  sm_mj       :\n{sm_mj}")
    print()

    # Robust sign extraction — same deep-unwrap pattern as for p_mj
    def deep_sign(v):
        """Unwrap nested structures to get a ±1 int."""
        if isinstance(v, (int, float)):
            return int(v)
        try:
            return int(float(v))
        except (TypeError, ValueError):
            pass
        if hasattr(v, 'values'):
            return deep_sign(next(iter(v.values())))
        if hasattr(v, '__iter__') and not isinstance(v, str):
            return deep_sign(next(iter(v)))
        raise TypeError(f"Cannot extract sign from {type(v)}: {v!r}")

    # Try same five strategies as get_junction_participation
    import pandas as pd
    sign_val = None
    for _strategy, _get in [
        ("A: iloc[0]['rect_jj1']",    lambda: sm_mj.iloc[0]['rect_jj1']),
        ("B: iloc[0, 0]",             lambda: sm_mj.iloc[0, 0]),
        ("C: loc['rect_jj1'].iloc[0]",lambda: sm_mj.loc['rect_jj1'].iloc[0]),
        ("D: inner iloc[0]",          lambda: (sm_mj.iloc[0]['rect_jj1']
                                               if isinstance(sm_mj.iloc[0], pd.DataFrame)
                                               else None)),
    ]:
        try:
            raw = _get()
            if raw is not None:
                sign_val = deep_sign(raw)
                print(f"  Strategy {_strategy} succeeded → sign = {sign_val}")
                break
        except Exception:
            pass

    if sign_val is None:
        print("  Could not extract sign numerically — read from run_epr() output above.")
        print("  Look for: 'sign s_0j    (+)'")
    else:
        sign_str = "+1  ✓" if sign_val > 0 else "-1  ✗  (check JJ LineString direction in HFSS)"
        print(f"\n  rect_jj1  s_0j = {sign_str}")

except Exception as ex:
    print(f"  [sm_mj not available: {ex}]")
    print("  PSR is printed in the run_epr() output above.")
    print("  Look for: 'sign s_0j    (+)'")

print("""
  Interpretation:
    s = +1 : JJ current flows in the positive sense — correct orientation.
    s = -1 : reversed polarity; check that the rect_jj1 LineString in
             make_jj() points outward (island → ground plane).

  Expected for TC1 (CircmonG, jj_angle=0): rect_jj1 s_0j = +1.
""")


In [ ]:
# -- Final design summary -----------------------------------------------------
print("=" * 70)
print("  DESIGN SUMMARY  -- CircmonG (Single-JJ Circular Grounded Transmon)")
print("=" * 70)

print(f"\n  Geometry (CircmonG 'TC1')")
print(f"    pad_r      : {circmon1.options.pad_r}  (qubit island radius)")
print(f"    pad2pk_gap : {circmon1.options.pad2pk_gap}  (circular slot width)")
print(f"    jj_width   : {circmon1.options.jj_width}")
print(f"    jj_angle   : {circmon1.options.jj_angle} deg")

print(f"\n  HFSS simulation")
print(f"    Lj (single JJ) : {eig_qb.sim.setup.vars.Lj}")
print(f"    Cj             : {eig_qb.sim.setup.vars.Cj}")
print(f"    Bare f         : {f_hfss_GHz:.4f} GHz")

try:
    print(f"\n  Quantum Hamiltonian")
    print(f"    f_01     : {float(f_ND[0]):.4f} GHz")
    print(f"    Ec       : {Ec_ND*1e3:.2f} MHz")
    print(f"    alpha    : {alpha_ND*1e3:.2f} MHz")
    print(f"    Ej       : {Ej_eff_qa:.4f} GHz  (single JJ)")
    print(f"    Ej/Ec    : {EjEc:.1f}")
except NameError:
    print("    (Complete Step 9 to populate Hamiltonian parameters)")

print(f"\n  Bulk Dielectric Loss")
print(f"    p_sub    : {p_sub:.4f}  ({p_sub*100:.1f}%)")
print(f"    T1_bulk  : {T1_diel_us:.0f} us  (tan_d_Si={tan_delta_Si:.0e})")

print(f"\n  Surface Participation Ratios (SPR)")
try:
    for key, res in spr_results.items():
        sp   = surfaces[key]
        flag = ' *' if 'proxy' in res['method'] else ''
        print(f"    {sp['label']:35s}: p = {res['p_s']:.3e}  T1 = {res['T1_us']:.0f} us{flag}")
    print(f"    (* = proxy estimate; re-run with HFSS field calc for accuracy)")
    print(f"\n  T1 Budget")
    print(f"    T1_total (SPR + bulk) : {T1_total_us:.0f} us  ({T1_total_us/1000:.2f} ms)")
    print(f"    Dominant loss         : {dominant_key}  (T1 = {dominant_T1:.0f} us)")
except NameError:
    print("    (Complete Step 8b to populate SPR results)")

try:
    ok_T1 = 'PASS' if T1_total_us >= 500 else 'NEEDS WORK'
    ok_f  = 'PASS' if 3.0 <= float(f_ND[0]) <= 4.8 else 'FAIL'
    ok_ec = 'PASS' if 120 <= Ec_ND*1e3 <= 160 else 'FAIL'
    ok_ej = 'PASS' if EjEc > 50 else 'FAIL'
    print(f"\n  Target summary")
    print(f"    f_01 in [3, 4.8] GHz : {float(f_ND[0]):.4f} GHz  [{ok_f}]")
    print(f"    Ec  in [120,160] MHz : {Ec_ND*1e3:.1f} MHz    [{ok_ec}]")
    print(f"    Ej/Ec > 50           : {EjEc:.1f}            [{ok_ej}]")
    print(f"    T1 > 500 us          : {T1_total_us:.0f} us            [{ok_T1}]")
except NameError:
    print("    (Complete Steps 7-9 to populate targets)")

print("=" * 70)


---
## Tuning Guide

| Goal | Action | Effect |
|---|---|---|
| ↑ f_01 | Decrease `Lj` | ↑ Ej → ↑ f |
| ↓ f_01 | Increase `Lj` | ↓ Ej → ↓ f |
| ↑ Ec (more anharmonic) | Decrease `Rdisk` | ↓ CΣ → ↑ Ec |
| ↓ Ec | Increase `Rdisk` or `Wslot` | ↑ CΣ → ↓ Ec |
| Reduce substrate loss | Decrease `W` (smaller ground plane area) | Less substrate E-field |
| Reduce MA surface loss | Increase `Rdisk` + `W` | E-field more dilute over larger area |
| Stronger isolation | Increase `W` | More ground plane around the qubit |

**Lj convention**: Each JJ is assigned inductance `Lj` in HFSS.
The two JJs in parallel give `Lj_eff = Lj/2` and `Ej_eff = 2 * (Phi0/2pi)^2 / Lj`.


In [ ]:
# ── Optional: close HFSS ──────────────────────────────────────────────────
eig_qb.sim.close()
gui.main_window.close()
print("Notebook complete.")
print("Uncomment the two lines above to close HFSS and the Metal GUI.")
# ── Optional: close HFSS ──────────────────────────────────────────────────
import gc

# Release field overlay COM objects from plot_fields() calls
try:
    eig_qb.sim.clear_fields()
except Exception:
    pass

# Release setup/solution COM handles held by the distributed analysis
try:
    da = eig_qb.sim.renderer.epr_distributed_analysis
    del da
except Exception:
    pass

# Force Python GC to decrement all remaining COM reference counts
gc.collect()
gc.collect()
# Now safe to close
eig_qb.sim.close()
gui.main_window.close()
print("Notebook complete.")
print("Uncomment the two lines above to close HFSS and the Metal GUI.")

